# HC3 Cleaning Pipeline (v2)

End-to-end, reproducible cleaning of the HC3 Wiki-CSAI and Finance subsets
for AI vs human text analysis.

**Design choices over v1:**

1. Filters ChatGPT boilerplate/error strings that contaminated the AI class.
2. Percentile-based length floor instead of an arbitrary 50-character cut.
3. Pair integrity enforced before sampling, so every retained `qid` has
   both a human and an AI answer (paired tests in Section 3 become valid).
4. Stratified random sampling with seeded `random_state` instead of
   `.first()`, which silently discarded ~570 finance AI answers.
5. Stylometric features added at cleaning time: burstiness, type-token
   ratio, comma density. These are the inputs for Section 3 visualisation
   and Section 4 modelling.
6. A `cleaning_log` sheet records rows in/out at every step for full
   reproducibility.


In [1]:
import re
import pandas as pd

CLEANING_LOG = []

def log(step, rows, note=""):
    CLEANING_LOG.append({"step": step, "rows": rows, "note": note})
    print(f"{step:>40} | rows={rows:>6} | {note}")


## Step 1. Load raw JSONL files

In [2]:
wiki = pd.read_json("wiki_csai.jsonl", lines=True)
wiki["source"] = "wiki_csai"

finance = pd.read_json("finance.jsonl", lines=True)
finance["source"] = "finance"

hc3 = pd.concat([wiki, finance], ignore_index=True)
hc3["qid"] = hc3["source"] + "_" + hc3.index.astype(str)
log("01_load_raw_questions", len(hc3), "wiki=842, finance=3933")


                   01_load_raw_questions | rows=  4775 | wiki=842, finance=3933


## Step 2. Explode to long format

Each question may carry more than one ChatGPT answer (finance has 570 such
cases). Exploding now preserves every answer; we balance later, deliberately.

In [3]:
human = (hc3[["qid", "question", "source", "human_answers"]]
         .explode("human_answers")
         .rename(columns={"human_answers": "text"})
         .assign(label="human"))

ai = (hc3[["qid", "question", "source", "chatgpt_answers"]]
      .explode("chatgpt_answers")
      .rename(columns={"chatgpt_answers": "text"})
      .assign(label="ai"))

df = pd.concat([human, ai], ignore_index=True).dropna(subset=["text"]).reset_index(drop=True)
log("02_explode_long", len(df),
    f"human={(df.label=='human').sum()}, ai={(df.label=='ai').sum()}")


                         02_explode_long | rows= 10120 | human=4775, ai=5345


## Step 3. Drop ChatGPT boilerplate and error strings

The original cleaning left ~30 rate-limit and refusal strings in the AI
class. These are not answers, they are platform artefacts. Removing them
prevents inflated, spurious class signal.

In [4]:
JUNK_PATTERNS = [
    r"^\s*!\s*Only one message at a time",
    r"^\s*I'?m sorry,?\s*but as an AI",
    r"^\s*As an AI language model",
    r"^\s*I (?:cannot|can'?t) (?:provide|access|browse|generate)",
    r"^\s*I do not have (?:access|the ability)",
    r"^\s*Sorry,? I cannot",
    r"There was an error generating a response",
]

junk_mask = df["text"].str.contains("|".join(JUNK_PATTERNS),
                                    regex=True, case=False, na=False)
df = df[~junk_mask].reset_index(drop=True)
log("03_drop_junk_strings", len(df), f"removed {junk_mask.sum()} boilerplate rows")


                    03_drop_junk_strings | rows= 10087 | removed 33 boilerplate rows


## Step 4. Normalise text

Strip URL placeholders, raw URLs, control characters, and collapse
whitespace. Also remove the legacy ELI5 prompt suffix from questions.

In [5]:
df["question"] = (df["question"]
    .str.replace(r"\s*Please explain like I'?m five\.?\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

df["text"] = (df["text"]
    .str.replace(r"URL_\d+", "", regex=True)
    .str.replace(r"http\S+|www\.\S+", "", regex=True)
    .str.replace(r"[\x00-\x1f\x7f]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

log("04_normalise_text", len(df), "URLs, whitespace, control chars")


                       04_normalise_text | rows= 10087 | URLs, whitespace, control chars


## Step 5. Deduplicate

In [6]:
before = len(df)
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
log("05_drop_text_duplicates", len(df), f"removed {before - len(df)}")


                 05_drop_text_duplicates | rows= 10003 | removed 84


## Step 6. Length floor (percentile-based, not arbitrary)

Drop the bottom 5% by word count. This removes one-line stubs without
imposing an unjustified magic number, and is defensible in writing.

In [7]:
df["word_count"] = df["text"].str.split().str.len()
floor = int(df["word_count"].quantile(0.05))

before = len(df)
df = df[df["word_count"] >= floor].reset_index(drop=True)
log("06_length_floor", len(df), f"floor=p05={floor} words, removed {before - len(df)}")


                         06_length_floor | rows=  9504 | floor=p05=40 words, removed 499


## Step 7. Enforce pair integrity

For Section 3 paired statistical tests (Wilcoxon, paired t-test) to be
valid, every `qid` must still have both a human and an AI answer.

In [8]:
label_per_qid = df.groupby("qid")["label"].nunique()
valid_qids = label_per_qid[label_per_qid == 2].index

before = len(df)
df = df[df["qid"].isin(valid_qids)].reset_index(drop=True)
log("07_enforce_pair_integrity", len(df), f"removed {before - len(df)} unmatched rows")


               07_enforce_pair_integrity | rows=  8908 | removed 596 unmatched rows


## Step 8. Stratified rebalance, one answer per (qid, label)

Where multiple AI answers exist for a question (finance has 570 such
cases), sample one with a seeded RNG. Reproducible and explicit.

In [9]:
RNG = 42
df = (df
      .groupby(["qid", "label"], as_index=False)
      .sample(n=1, random_state=RNG)
      .reset_index(drop=True))
log("08_stratified_sample", len(df), f"random_state={RNG}")


                    08_stratified_sample | rows=  8422 | random_state=42


## Step 9. Feature engineering for Section 3

Three stylometric features that, in the literature, separate LLM output
from human writing:

- **Burstiness**: std / mean of sentence length in words. Low burstiness
  is a documented LLM signature.
- **Type-token ratio**: unique tokens / total tokens. A proxy for lexical
  diversity.
- **Comma density**: commas per sentence. LLMs tend to be syntactically
  more elaborated.


In [10]:
df["char_count"] = df["text"].str.len()
df["sentence_count"] = df["text"].str.count(r"[.!?]+").clip(lower=1)
df["avg_word_length"] = (df["char_count"] - df["word_count"] + 1) / df["word_count"]
df["avg_sentence_length_words"] = df["word_count"] / df["sentence_count"]

def burstiness(text):
    sents = re.split(r"[.!?]+", text)
    lens = [len(s.split()) for s in sents if s.strip()]
    if len(lens) < 2:
        return 0.0
    s = pd.Series(lens)
    m = s.mean()
    return float(s.std() / m) if m > 0 else 0.0

def type_token_ratio(text):
    toks = re.findall(r"[A-Za-z']+", text.lower())
    return len(set(toks)) / len(toks) if toks else 0.0

df["burstiness"] = df["text"].apply(burstiness)
df["type_token_ratio"] = df["text"].apply(type_token_ratio)
df["comma_density"] = df["text"].str.count(",") / df["sentence_count"]

log("09_features_built", len(df), "burstiness, TTR, comma density added")


                       09_features_built | rows=  8422 | burstiness, TTR, comma density added


## Step 10. Persist outputs

Three sheets in one xlsx: clean data, cleaning log, summary by class.

In [11]:
cols = ["qid", "label", "source", "question", "text",
        "char_count", "word_count", "sentence_count",
        "avg_word_length", "avg_sentence_length_words",
        "burstiness", "type_token_ratio", "comma_density"]
df = df[cols]

cleaning_log = pd.DataFrame(CLEANING_LOG)

summary = (df.groupby(["source", "label"])
             [["char_count", "word_count", "sentence_count",
               "burstiness", "type_token_ratio", "comma_density"]]
             .agg(["mean", "median", "std"])
             .round(3))

with pd.ExcelWriter("hc3_clean_v2.xlsx", engine="openpyxl") as xl:
    df.to_excel(xl, sheet_name="data", index=False)
    cleaning_log.to_excel(xl, sheet_name="cleaning_log", index=False)
    summary.to_excel(xl, sheet_name="summary_by_class")

print("Final shape:", df.shape)
print("Pair integrity:", (df.groupby("qid")["label"].nunique() == 2).all())


Final shape: (8422, 13)
Pair integrity: True


## Sanity check: signal preview

The features already separate the two classes. These are the headline
findings Section 3 should visualise.

In [12]:
df.groupby("label")[["word_count", "burstiness",
                       "type_token_ratio", "comma_density"]].mean().round(3)


,word_count,burstiness,type_token_ratio,comma_density
label,,,,
ai,205.156,0.337,0.49,1.115
human,193.696,0.539,0.62,0.952
